# 07 - SOTA Fine-Tuning: EfficientNet-B0

Fine-tune an ImageNet-pretrained **EfficientNet-B0** on the exact Intel Image
Classification train/validation/test split prepared in
`01_data_preparation.ipynb`.

This notebook follows the same output conventions as the other model
notebooks and uses **64x64 RGB inputs** for a fair resource-constrained
comparison. Model selection and fine-tuning decisions use validation data
only; the held-out test set is evaluated once after the final checkpoint has
been selected.


## 1. Environment Setup


In [ ]:
import os

# Request deterministic TensorFlow operations before importing TensorFlow.
os.environ.setdefault("TF_DETERMINISTIC_OPS", "1")

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount("/content/drive")
    PROJECT_ROOT = "/content/drive/MyDrive/EN3150-A03-Resource-Constrained-CNN"
else:
    # When run from the repository's notebooks/ directory, this resolves to
    # the repository root. Override with EN3150_PROJECT_ROOT when necessary.
    PROJECT_ROOT = os.environ.get("EN3150_PROJECT_ROOT", os.path.abspath(".."))

print("IN_COLAB:", IN_COLAB)
print("PROJECT_ROOT:", PROJECT_ROOT)


## 2. Configuration and Paths


In [ ]:
CONFIG = {
    "image_size": 64,
    "channels": 3,
    "random_seed": 42,
    "batch_size": 32,
    "epochs_frozen": 8,
    "epochs_fine_tune": 12,
    # Fine-tune complete residual blocks 6 and 7 plus the top convolution.
    # A block boundary is safer than cutting at an arbitrary layer fraction.
    "fine_tune_from_block": "block6a",
    "lr_frozen": 1e-3,
    "lr_fine_tune": 1e-5,
    "dropout_rate": 0.20,
    "dataset_name": "Intel Image Classification",
    "num_classes": 6,
    "class_names": [
        "buildings", "forest", "glacier", "mountain", "sea", "street"
    ],
}
CONFIG["epochs"] = CONFIG["epochs_frozen"] + CONFIG["epochs_fine_tune"]
assert CONFIG["epochs"] >= 20, "The assignment requires at least 20 epochs."
assert CONFIG["image_size"] <= 64, "Input resolution must not exceed 64x64."

# Notebook 01 has previously been run under two Drive-root layouts. Search
# both so this notebook consumes the authoritative cached split instead of
# silently reprocessing or re-splitting the dataset.
PROCESSED_SUBPATH = os.path.join(
    "data", "processed", "intel-image-classification"
)
CANDIDATE_ROOTS = [
    PROJECT_ROOT,
    os.path.join(PROJECT_ROOT, "EN3150-A03-Resource-Constrained-CNN"),
]
PROCESSED_DIR = os.path.join(PROJECT_ROOT, PROCESSED_SUBPATH)
for candidate_root in CANDIDATE_ROOTS:
    candidate_dir = os.path.join(candidate_root, PROCESSED_SUBPATH)
    if os.path.exists(os.path.join(candidate_dir, "X_train.npy")):
        PROCESSED_DIR = candidate_dir
        break

CHECKPOINT_DIR = os.path.join(
    PROJECT_ROOT, "models", "checkpoints", "efficientnet"
)
FINAL_MODEL_DIR = os.path.join(PROJECT_ROOT, "models", "final")
FIGURES_DIR = os.path.join(
    PROJECT_ROOT, "results", "figures", "efficientnet"
)
CONFUSION_DIR = os.path.join(
    PROJECT_ROOT, "results", "confusion_matrices"
)
METRICS_DIR = os.path.join(PROJECT_ROOT, "results", "metrics")
TABLES_DIR = os.path.join(PROJECT_ROOT, "results", "tables")

for directory in (
    CHECKPOINT_DIR,
    FINAL_MODEL_DIR,
    FIGURES_DIR,
    CONFUSION_DIR,
    METRICS_DIR,
    TABLES_DIR,
):
    os.makedirs(directory, exist_ok=True)

print("PROCESSED_DIR:", PROCESSED_DIR)
CONFIG


## 3. Imports and Reproducibility


In [ ]:
import json
import platform
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
)
from tensorflow import keras
from tensorflow.keras.applications import EfficientNetB0

SEED = CONFIG["random_seed"]
keras.utils.set_random_seed(SEED)
try:
    tf.config.experimental.enable_op_determinism()
    DETERMINISTIC_OPS_ENABLED = True
except Exception as exc:
    # Some older TensorFlow builds do not expose this function. Seeds are
    # still fixed, and the limitation is recorded in the metrics file.
    DETERMINISTIC_OPS_ENABLED = False
    print("Deterministic-op warning:", exc)

print("Python:", platform.python_version())
print("TensorFlow:", tf.__version__)
print("GPU devices:", tf.config.list_physical_devices("GPU"))
print("Deterministic ops enabled:", DETERMINISTIC_OPS_ENABLED)


## 4. Load and Validate the Prepared Data


Notebook 01 stores the common arrays as `float32` values in **[0, 1]**.
EfficientNet's Keras implementation already contains its own `Rescaling`
layer and expects float pixels in **[0, 255]**. Therefore the model below
contains one explicit `Rescaling(255.0)` layer before the pretrained
backbone. No additional `preprocess_input` call is used.


In [ ]:
required_files = [
    "X_train.npy", "y_train.npy",
    "X_val.npy", "y_val.npy",
    "X_test.npy", "y_test.npy",
    "class_names.json",
]
missing_files = [
    name for name in required_files
    if not os.path.exists(os.path.join(PROCESSED_DIR, name))
]
if missing_files:
    raise FileNotFoundError(
        "Notebook 01 artifacts are missing from "
        f"{PROCESSED_DIR}: {missing_files}"
    )

X_train = np.load(os.path.join(PROCESSED_DIR, "X_train.npy"), mmap_mode="r")
y_train = np.load(os.path.join(PROCESSED_DIR, "y_train.npy"), mmap_mode="r")
X_val = np.load(os.path.join(PROCESSED_DIR, "X_val.npy"), mmap_mode="r")
y_val = np.load(os.path.join(PROCESSED_DIR, "y_val.npy"), mmap_mode="r")
X_test = np.load(os.path.join(PROCESSED_DIR, "X_test.npy"), mmap_mode="r")
y_test = np.load(os.path.join(PROCESSED_DIR, "y_test.npy"), mmap_mode="r")

with open(os.path.join(PROCESSED_DIR, "class_names.json"), encoding="utf-8") as file:
    class_names = json.load(file)

expected_image_shape = (
    CONFIG["image_size"], CONFIG["image_size"], CONFIG["channels"]
)
assert class_names == CONFIG["class_names"], (
    "class_names.json does not match the agreed class order."
)
for split_name, X, y in (
    ("train", X_train, y_train),
    ("validation", X_val, y_val),
    ("test", X_test, y_test),
):
    assert len(X) == len(y), f"{split_name}: image/label counts differ"
    assert X.shape[1:] == expected_image_shape, (
        f"{split_name}: expected {expected_image_shape}, got {X.shape[1:]}"
    )
    assert X.dtype == np.float32, f"{split_name}: expected float32, got {X.dtype}"
    assert float(X.min()) >= 0.0 and float(X.max()) <= 1.0, (
        f"{split_name}: expected normalized pixels in [0, 1]"
    )
    assert np.issubdtype(y.dtype, np.integer), (
        f"{split_name}: labels must be integer encoded"
    )
    assert set(np.unique(y)).issubset(range(CONFIG["num_classes"])), (
        f"{split_name}: labels outside [0, {CONFIG['num_classes'] - 1}]"
    )

n_total = len(X_train) + len(X_val) + len(X_test)
split_ratios = np.array([len(X_train), len(X_val), len(X_test)]) / n_total
assert np.allclose(split_ratios, [0.70, 0.15, 0.15], atol=0.001), split_ratios

print("Train:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)
print("Test:", X_test.shape, y_test.shape)
print("Split ratios:", split_ratios.round(4).tolist())
print("Pixel range:", float(X_train.min()), "to", float(X_train.max()))
print("Class order:", class_names)


## 5. Build Deterministic `tf.data` Pipelines


In [ ]:
AUTOTUNE = tf.data.AUTOTUNE


def make_dataset(X, y, training):
    ds = tf.data.Dataset.from_tensor_slices((X, y))
    if training:
        ds = ds.shuffle(
            buffer_size=len(X),
            seed=SEED,
            reshuffle_each_iteration=True,
        )
    ds = ds.map(
        lambda image, label: (
            tf.cast(image, tf.float32),
            tf.cast(label, tf.int32),
        ),
        num_parallel_calls=AUTOTUNE,
        deterministic=True,
    )
    ds = ds.batch(CONFIG["batch_size"], drop_remainder=False)
    return ds.prefetch(AUTOTUNE)


train_ds = make_dataset(X_train, y_train, training=True)
val_ds = make_dataset(X_val, y_val, training=False)
test_ds = make_dataset(X_test, y_test, training=False)

sample_images, sample_labels = next(iter(train_ds))
assert sample_images.shape[1:] == expected_image_shape
assert sample_images.dtype == tf.float32
print("Example batch:", sample_images.shape, sample_labels.shape)


## 6. Build EfficientNet-B0


In [ ]:
input_shape = (
    CONFIG["image_size"], CONFIG["image_size"], CONFIG["channels"]
)

data_augmentation = keras.Sequential(
    [
        keras.layers.RandomFlip("horizontal", seed=SEED),
        keras.layers.RandomRotation(0.05, seed=SEED + 1),
        keras.layers.RandomZoom(0.10, seed=SEED + 2),
    ],
    name="training_augmentation",
)

base_model = EfficientNetB0(
    input_shape=input_shape,
    include_top=False,
    weights="imagenet",
)
base_model.trainable = False

inputs = keras.Input(shape=input_shape, name="normalized_image")
x = data_augmentation(inputs)
# Project arrays are [0, 1], while Keras EfficientNet expects [0, 255] and
# performs its own internal normalization. This layer is part of the saved
# model, so inference uses the same preprocessing as training.
x = keras.layers.Rescaling(255.0, name="restore_0_255_range")(x)
# training=False keeps the pretrained backbone's BatchNorm statistics fixed.
# Gradients still update any non-BatchNorm layers later marked trainable.
x = base_model(x, training=False)
x = keras.layers.GlobalAveragePooling2D(name="global_average_pooling")(x)
x = keras.layers.Dropout(CONFIG["dropout_rate"], seed=SEED + 3)(x)
outputs = keras.layers.Dense(
    CONFIG["num_classes"], activation="softmax", name="classifier"
)(x)

model = keras.Model(inputs, outputs, name="efficientnetb0_intel_scenes")
model.summary()


## 7. Parameter Counts Before Fine-Tuning


In [ ]:
def count_weight_parameters(weights):
    return int(sum(np.prod(weight.shape) for weight in weights))


total_params = int(model.count_params())
trainable_params_phase1 = count_weight_parameters(model.trainable_weights)
non_trainable_params_phase1 = total_params - trainable_params_phase1
theoretical_fp32_weight_size_mb = total_params * 4 / (1024 ** 2)
weights_size_kb = total_params * 4 / 1024

print(f"Total parameters:                 {total_params:,}")
print(f"Trainable parameters (Phase 1):   {trainable_params_phase1:,}")
print(f"Non-trainable parameters (Phase 1): {non_trainable_params_phase1:,}")
print(f"Theoretical FP32 weight size:     {theoretical_fp32_weight_size_mb:.2f} MB")


## 8. MAC Operations

For compatibility with Model B, computational cost is reported as
multiply-accumulate operations (MACs) for one 64x64 image. The counter below
uses the same convention as Notebook 03: it includes `Conv2D`,
`DepthwiseConv2D`, and `Dense` operations, while excluding Batch
Normalization, activation, pooling, elementwise addition, augmentation, and
data-transfer overhead. One multiplication followed by its accumulation is
counted as one MAC.

Observed batch-1 latency is measured separately later because MAC count is an
architecture-level quantity, while latency also depends on the GPU/CPU and
software runtime.


In [ ]:
def iter_leaf_layers(layer):
    # Yield every non-Model layer, including layers inside nested models.
    if isinstance(layer, keras.Model):
        for child in layer.layers:
            yield from iter_leaf_layers(child)
    else:
        yield layer


def output_shape_tuple(layer):
    # Return a concrete single-output shape tuple for a built Keras layer.
    try:
        shape = layer.output_shape
    except AttributeError:
        shape = layer.output.shape
    if isinstance(shape, list):
        raise ValueError(f"Layer {layer.name!r} has multiple outputs")
    return tuple(None if dim is None else int(dim) for dim in shape)


def count_model_macs(model_to_count):
    rows = []
    for layer in iter_leaf_layers(model_to_count):
        macs = None
        if isinstance(layer, keras.layers.DepthwiseConv2D):
            out_shape = output_shape_tuple(layer)
            output_positions = int(np.prod(out_shape[1:-1]))
            macs = output_positions * int(np.prod(layer.depthwise_kernel.shape))
        elif isinstance(layer, keras.layers.Conv2D):
            out_shape = output_shape_tuple(layer)
            output_positions = int(np.prod(out_shape[1:-1]))
            # The kernel shape already accounts for grouped input channels.
            macs = output_positions * int(np.prod(layer.kernel.shape))
        elif isinstance(layer, keras.layers.Dense):
            macs = int(np.prod(layer.kernel.shape))

        if macs is not None:
            rows.append({
                "layer": layer.name,
                "type": layer.__class__.__name__,
                "output_shape": str(output_shape_tuple(layer)),
                "macs": int(macs),
            })

    breakdown = pd.DataFrame(rows)
    return int(breakdown["macs"].sum()), breakdown


total_macs, macs_breakdown = count_model_macs(model)
macs_table_path = os.path.join(TABLES_DIR, "efficientnet_macs.csv")
macs_breakdown.to_csv(macs_table_path, index=False)

print(f"EfficientNet-B0 MACs per 64x64 image: {total_macs:,}")
print(f"Counted convolution/dense layers:     {len(macs_breakdown)}")
print("Saved MAC breakdown:", macs_table_path)
macs_breakdown.groupby("type", as_index=False)["macs"].sum()


## 9. Fine-Tuning Strategy

Training uses two phases totaling 20 epochs:

1. **Frozen feature extractor (8 epochs).** Only the new six-class head is
   trained with Adam at (10^{-3}).
2. **Partial fine-tuning (12 epochs).** The best Phase-1 checkpoint is
   restored. Complete EfficientNet blocks 6 and 7 plus the top convolution
   are eligible for training, while all Batch Normalization layers remain
   frozen. Using a named block boundary avoids splitting a residual block.
   Adam then uses the much smaller learning rate (10^{-5}).

Separate phase checkpoints are used. After both phases, the checkpoint with
the lower best **validation loss** is selected. The test set is not consulted
during model selection.


## 10. Phase 1 - Train the Classification Head


In [ ]:
phase1_checkpoint = os.path.join(CHECKPOINT_DIR, "phase1_best.keras")
phase1_callback = keras.callbacks.ModelCheckpoint(
    phase1_checkpoint,
    monitor="val_loss",
    mode="min",
    save_best_only=True,
    verbose=1,
)


class EpochTimer(keras.callbacks.Callback):
    # Record wall-clock duration for each completed epoch.

    def on_train_begin(self, logs=None):
        self.times = []

    def on_epoch_begin(self, epoch, logs=None):
        self.epoch_start = time.perf_counter()

    def on_epoch_end(self, epoch, logs=None):
        self.times.append(float(time.perf_counter() - self.epoch_start))


phase1_timer = EpochTimer()

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=CONFIG["lr_frozen"]),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

phase1_start = time.perf_counter()
history_phase1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=CONFIG["epochs_frozen"],
    callbacks=[phase1_callback, phase1_timer],
    verbose=1,
)
phase1_time_sec = time.perf_counter() - phase1_start
print(
    f"Phase 1: {phase1_time_sec:.1f}s total, "
    f"{phase1_time_sec / CONFIG['epochs_frozen']:.1f}s/epoch"
)


## 11. Phase 2 - Partially Fine-Tune the Backbone


In [ ]:
# Start fine-tuning from the best Phase-1 checkpoint, not the last epoch.
model = keras.models.load_model(phase1_checkpoint)
base_model = model.get_layer("efficientnetb0")
base_model.trainable = True

fine_tune_prefix = CONFIG["fine_tune_from_block"]
fine_tune_candidates = [
    index
    for index, layer in enumerate(base_model.layers)
    if layer.name.startswith(fine_tune_prefix)
]
if not fine_tune_candidates:
    raise ValueError(
        f"Could not find EfficientNet block prefix {fine_tune_prefix!r}. "
        "Inspect base_model layer names before changing TensorFlow versions."
    )
fine_tune_at = fine_tune_candidates[0]
for index, layer in enumerate(base_model.layers):
    should_train = index >= fine_tune_at
    # Preserve ImageNet BatchNorm moving statistics for the small target data.
    if isinstance(layer, keras.layers.BatchNormalization):
        should_train = False
    layer.trainable = should_train

trainable_backbone_layers = [
    layer.name for layer in base_model.layers if layer.trainable
]
trainable_params_phase2 = count_weight_parameters(model.trainable_weights)
non_trainable_params_phase2 = total_params - trainable_params_phase2

print(
    f"Fine-tuning from backbone layer index {fine_tune_at} of "
    f"{len(base_model.layers)}."
)
print("Trainable backbone layers:", len(trainable_backbone_layers))
print(f"Trainable parameters (Phase 2): {trainable_params_phase2:,}")
print(f"Non-trainable parameters (Phase 2): {non_trainable_params_phase2:,}")

phase2_checkpoint = os.path.join(CHECKPOINT_DIR, "phase2_best.keras")
phase2_callback = keras.callbacks.ModelCheckpoint(
    phase2_checkpoint,
    monitor="val_loss",
    mode="min",
    save_best_only=True,
    verbose=1,
)
phase2_timer = EpochTimer()

# Recompile after changing trainable flags.
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=CONFIG["lr_fine_tune"]),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

phase2_start = time.perf_counter()
history_phase2 = model.fit(
    train_ds,
    validation_data=val_ds,
    initial_epoch=CONFIG["epochs_frozen"],
    epochs=CONFIG["epochs"],
    callbacks=[phase2_callback, phase2_timer],
    verbose=1,
)
phase2_time_sec = time.perf_counter() - phase2_start
print(
    f"Phase 2: {phase2_time_sec:.1f}s total, "
    f"{phase2_time_sec / CONFIG['epochs_fine_tune']:.1f}s/epoch"
)


## 12. Select the Best Validation Checkpoint


In [ ]:
best_phase1_val_loss = float(min(history_phase1.history["val_loss"]))
best_phase2_val_loss = float(min(history_phase2.history["val_loss"]))
best_phase1_epoch = int(np.argmin(history_phase1.history["val_loss"]) + 1)
best_phase2_epoch = int(
    np.argmin(history_phase2.history["val_loss"])
    + CONFIG["epochs_frozen"]
    + 1
)

if best_phase2_val_loss < best_phase1_val_loss:
    selected_phase = "phase2_fine_tuned"
    selected_checkpoint = phase2_checkpoint
    selected_best_val_loss = best_phase2_val_loss
    selected_best_epoch = best_phase2_epoch
else:
    selected_phase = "phase1_frozen_backbone"
    selected_checkpoint = phase1_checkpoint
    selected_best_val_loss = best_phase1_val_loss
    selected_best_epoch = best_phase1_epoch

model = keras.models.load_model(selected_checkpoint)
print("Selected checkpoint:", selected_checkpoint)
print("Selected phase:", selected_phase)
print("Best validation loss:", selected_best_val_loss)
print("Best epoch:", selected_best_epoch)


## 13. Training and Validation Curves


In [ ]:
def combined_history(key):
    return [
        float(value)
        for value in (
            history_phase1.history[key] + history_phase2.history[key]
        )
    ]


train_loss = combined_history("loss")
val_loss = combined_history("val_loss")
train_accuracy = combined_history("accuracy")
val_accuracy = combined_history("val_accuracy")
epoch_numbers = np.arange(1, CONFIG["epochs"] + 1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].plot(epoch_numbers, train_loss, label="Training")
axes[0].plot(epoch_numbers, val_loss, label="Validation")
axes[0].axvline(
    CONFIG["epochs_frozen"] + 0.5,
    color="gray",
    linestyle="--",
    label="Fine-tuning starts",
)
axes[0].scatter(
    [selected_best_epoch],
    [selected_best_val_loss],
    color="red",
    zorder=3,
    label="Selected checkpoint",
)
axes[0].set(title="Loss", xlabel="Epoch", ylabel="Cross-entropy loss")
axes[0].legend()

axes[1].plot(epoch_numbers, train_accuracy, label="Training")
axes[1].plot(epoch_numbers, val_accuracy, label="Validation")
axes[1].axvline(
    CONFIG["epochs_frozen"] + 0.5,
    color="gray",
    linestyle="--",
    label="Fine-tuning starts",
)
axes[1].set(title="Accuracy", xlabel="Epoch", ylabel="Accuracy", ylim=(0, 1))
axes[1].legend()

fig.suptitle("EfficientNet-B0: Frozen Head Training and Partial Fine-Tuning")
fig.tight_layout()
curve_path = os.path.join(FIGURES_DIR, "training_curves.png")
fig.savefig(curve_path, dpi=150, bbox_inches="tight")
plt.show()
print("Saved:", curve_path)


## 14. Held-Out Test Evaluation


In [ ]:
# This is the first use of the held-out test labels for model assessment.
y_probability = model.predict(test_ds, verbose=1)
y_pred = np.argmax(y_probability, axis=1)

test_accuracy = accuracy_score(y_test, y_pred)
test_precision_macro = precision_score(
    y_test, y_pred, average="macro", zero_division=0
)
test_recall_macro = recall_score(
    y_test, y_pred, average="macro", zero_division=0
)
cm = confusion_matrix(
    y_test,
    y_pred,
    labels=np.arange(CONFIG["num_classes"]),
)
report_dict = classification_report(
    y_test,
    y_pred,
    labels=np.arange(CONFIG["num_classes"]),
    target_names=class_names,
    output_dict=True,
    zero_division=0,
)

print(f"Test accuracy:          {test_accuracy:.4f}")
print(f"Macro precision:        {test_precision_macro:.4f}")
print(f"Macro recall:           {test_recall_macro:.4f}")
print()
print(
    classification_report(
        y_test,
        y_pred,
        labels=np.arange(CONFIG["num_classes"]),
        target_names=class_names,
        zero_division=0,
    )
)

report_path = os.path.join(TABLES_DIR, "efficientnet_classification_report.csv")
pd.DataFrame(report_dict).transpose().to_csv(report_path)
print("Saved:", report_path)


## 15. Confusion Matrix


In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names,
    ax=ax,
)
ax.set_xlabel("Predicted class")
ax.set_ylabel("True class")
ax.set_title("EfficientNet-B0 - Test Confusion Matrix")
fig.tight_layout()
confusion_path = os.path.join(CONFUSION_DIR, "efficientnet.png")
fig.savefig(confusion_path, dpi=150, bbox_inches="tight")
plt.show()
print("Saved:", confusion_path)


## 16. Model Size and Inference Latency


In [ ]:
final_model_path = os.path.join(FINAL_MODEL_DIR, "efficientnet.keras")
model.save(final_model_path)
model_size_bytes = int(os.path.getsize(final_model_path))
model_size_kb = model_size_bytes / 1024
model_size_mb = model_size_bytes / (1024 ** 2)


def benchmark_single_image_latency(model_to_test, images, warmup=10, runs=50):
    sample = tf.convert_to_tensor(np.asarray(images[:1]), dtype=tf.float32)
    for _ in range(warmup):
        model_to_test(sample, training=False).numpy()

    timings_ms = []
    for _ in range(runs):
        start = time.perf_counter()
        model_to_test(sample, training=False).numpy()  # .numpy() synchronizes GPU work
        timings_ms.append((time.perf_counter() - start) * 1000.0)
    return {
        "runs": runs,
        "mean_ms_per_image": float(np.mean(timings_ms)),
        "median_ms_per_image": float(np.median(timings_ms)),
        "std_ms_per_image": float(np.std(timings_ms)),
    }


latency = benchmark_single_image_latency(model, X_test)
phase1_epoch_times = [float(value) for value in phase1_timer.times]
phase2_epoch_times = [float(value) for value in phase2_timer.times]
time_per_epoch_sec = phase1_epoch_times + phase2_epoch_times
assert len(time_per_epoch_sec) == CONFIG["epochs"]
avg_time_per_epoch_sec = (
    phase1_time_sec + phase2_time_sec
) / CONFIG["epochs"]

print("Saved final model:", final_model_path)
print(f"Saved .keras size: {model_size_mb:.2f} MB")
print(f"Theoretical FP32 weight size: {theoretical_fp32_weight_size_mb:.2f} MB")
print(f"Average training time/epoch: {avg_time_per_epoch_sec:.2f}s")
print(
    f"Phase 1 epoch time: {np.mean(phase1_epoch_times):.2f}s mean; "
    f"Phase 2: {np.mean(phase2_epoch_times):.2f}s mean"
)
print(f"Median batch-1 latency: {latency['median_ms_per_image']:.3f} ms/image")


## 17. Save Standardized Results


In [ ]:
metrics = {
    "model_name": "EfficientNet-B0",
    "dataset_name": CONFIG["dataset_name"],
    "input_size": CONFIG["image_size"],
    "input_channels": CONFIG["channels"],
    "input_value_range": [0.0, 1.0],
    "backbone_expected_range": [0.0, 255.0],
    "test_accuracy": float(test_accuracy),
    "test_precision_macro": float(test_precision_macro),
    "test_recall_macro": float(test_recall_macro),
    "confusion_matrix": cm.tolist(),
    "classification_report": report_dict,
    "class_names": class_names,
    "total_params": total_params,
    # Generic alias used by Model B / Notebook 08; the phase-specific counts
    # below remain the authoritative transfer-learning breakdown.
    "trainable_params": trainable_params_phase2,
    "trainable_params_phase1": trainable_params_phase1,
    "non_trainable_params_phase1": non_trainable_params_phase1,
    "trainable_params_phase2": trainable_params_phase2,
    "non_trainable_params_phase2": non_trainable_params_phase2,
    "macs": total_macs,
    "weights_size_kb": float(weights_size_kb),
    "theoretical_fp32_weight_size_mb": float(theoretical_fp32_weight_size_mb),
    "model_size_bytes": model_size_bytes,
    "model_size_kb": float(model_size_kb),
    "model_size_mb": float(model_size_mb),
    "epochs": CONFIG["epochs"],
    "epochs_frozen": CONFIG["epochs_frozen"],
    "epochs_fine_tune": CONFIG["epochs_fine_tune"],
    "fine_tune_from_block": CONFIG["fine_tune_from_block"],
    "fine_tune_start_layer_index": fine_tune_at,
    "trainable_backbone_layer_fraction": (
        len(trainable_backbone_layers) / len(base_model.layers)
    ),
    "trainable_backbone_layer_count": len(trainable_backbone_layers),
    "learning_rate_frozen": CONFIG["lr_frozen"],
    "learning_rate_fine_tune": CONFIG["lr_fine_tune"],
    "selected_phase": selected_phase,
    "selected_best_epoch": selected_best_epoch,
    "selected_best_val_loss": selected_best_val_loss,
    "best_phase1_val_loss": best_phase1_val_loss,
    "best_phase2_val_loss": best_phase2_val_loss,
    "phase1_time_sec": float(phase1_time_sec),
    "phase2_time_sec": float(phase2_time_sec),
    "avg_time_per_epoch_sec": float(avg_time_per_epoch_sec),
    "time_per_epoch_sec": time_per_epoch_sec,
    "time_per_epoch_phase1_sec": phase1_epoch_times,
    "time_per_epoch_phase2_sec": phase2_epoch_times,
    "inference_latency": latency,
    "tensorflow_version": tf.__version__,
    "deterministic_ops_enabled": DETERMINISTIC_OPS_ENABLED,
    "history": {
        "loss": train_loss,
        "val_loss": val_loss,
        "accuracy": train_accuracy,
        "val_accuracy": val_accuracy,
    },
    "artifacts": {
        "selected_checkpoint": selected_checkpoint,
        "final_model": final_model_path,
        "training_curves": curve_path,
        "confusion_matrix_figure": confusion_path,
        "classification_report_csv": report_path,
        "macs_breakdown_csv": macs_table_path,
    },
}

metrics_path = os.path.join(METRICS_DIR, "efficientnet.json")
with open(metrics_path, "w", encoding="utf-8") as file:
    json.dump(metrics, file, indent=2)

print("Saved metrics:", metrics_path)


## 18. Result Summary and Interpretation


In [ ]:
print(
    f"EfficientNet-B0 ({CONFIG['image_size']}x{CONFIG['image_size']}):\n"
    f"  Selected phase:                 {selected_phase}\n"
    f"  Best validation epoch/loss:     {selected_best_epoch} / "
    f"{selected_best_val_loss:.4f}\n"
    f"  Test accuracy:                  {test_accuracy:.4f}\n"
    f"  Test precision/recall (macro):  {test_precision_macro:.4f} / "
    f"{test_recall_macro:.4f}\n"
    f"  Total parameters:               {total_params:,}\n"
    f"  Trainable parameters (Phase 2): {trainable_params_phase2:,}\n"
    f"  MACs per 64x64 image:           {total_macs:,}\n"
    f"  Saved model size:               {model_size_mb:.2f} MB\n"
    f"  Median batch-1 latency:         "
    f"{latency['median_ms_per_image']:.3f} ms/image"
)


After executing the notebook, interpret the printed results rather than making
claims in advance. In the report, discuss:

- whether partial fine-tuning improved validation loss over the frozen model;
- which scene classes were most frequently confused;
- whether the accuracy gain over the custom models justifies EfficientNet-B0's
  greater parameter count, saved size, and inference latency;
- the limitation that ImageNet pretraining used larger source images, whereas
  this controlled assignment evaluates every architecture using only 64x64
  inputs.
